In [2]:
import pandas as pd
import os
import requests
import urllib.robotparser as rp
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import random as rand
import pandas as pd
import re
import time
from itables import init_notebook_mode
# init_notebook_mode(all_interactive=True)
import numpy as np

c:\Users\gcich\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.12.922) or chardet (None)/charset_normalizer (2.1.1) doesn't match a supported version!
  warnings.warn(


In [3]:
extracted_data_output_path = os.path.join(os.getcwd(), '..', 'data', 'data_extracted')
teams_replacement_dict = {
    'Man City': 'Manchester City',
    'Man United': 'Manchester United',
    'Manchester Utd': 'Manchester United',
    'Luton Town': 'Luton',
    'Ipswich Town': 'Ipswich',
    "Nott'm Forest": "Nottingham Forest",
    "Nott'ham Forest": "Nottingham Forest",
    'Leicester City': 'Leicester',
    'Newcastle Utd': 'Newcastle',
    'Sheffield Utd': 'Sheffield United',
    'Stoke City': 'Stoke',
    'Swansea City': 'Swansea',
    'Cardiff City': 'Cardiff',
    'Norwich City': 'Norwich',
    'Leeds United': 'Leeds'
    
}  


Read base data from data_co_uk

In [ ]:
cols = [
    'date_time',
    'referee',
    'home_team',
    'away_team',
    'home_goals',
    'away_goals',
    'result',
    'home_goals_h1',
    'away_goals_h1',
    'result_h1',
    'home_shots',
    'away_shots',
    'home_shots_on_target',
    'away_shots_on_target',
    'home_corners',
    'away_corners',
    'home_fouls',
    'away_fouls',
    'home_yellow_cards',
    'away_yellow_cards',
    'home_red_cards',
    'away_red_cards',
    'home_win_odds',
    'draw_odds',
    'away_win_odds',
    'over_2_5_goals_odds',
    'under_2_5_goals_odds'
]
base_data = pd.DataFrame(columns=cols)
data_path = os.path.join(os.getcwd(), '..', 'data', 'data_co_uk')
for filename in os.listdir(data_path):
    if os.path.splitext(filename)[1] != '.csv':
        continue

    data = pd.read_csv(os.path.join(data_path,filename))
    data = data.rename(columns={
        'Referee': 'referee',
        'HomeTeam': 'home_team',
        'AwayTeam': 'away_team',
        'FTHG': 'home_goals',
        'FTAG': 'away_goals',
        'FTR': 'result',
        'HTHG': 'home_goals_h1',
        'HTAG': 'away_goals_h1',
        'HTR': 'result_h1',
        'HS': 'home_shots',
        'AS': 'away_shots',
        'HST': 'home_shots_on_target',
        'AST': 'away_shots_on_target',
        'HC': 'home_corners',
        'AC': 'away_corners',
        'HF': 'home_fouls',
        'AF': 'away_fouls',
        'HY': 'home_yellow_cards',
        'AY': 'away_yellow_cards',
        'HR': 'home_red_cards',
        'AR': 'away_red_cards'
    })
    
    if filename not in ['PL_17_18.csv', 'PL_18_19.csv']:
        data = data.rename(columns={
            'AvgH': 'home_win_odds',
            'AvgD':	'draw_odds',
            'AvgA': 'away_win_odds',
            'Avg>2.5': 'over_2_5_goals_odds',
            'Avg<2.5': 'under_2_5_goals_odds'
        })
    else:
        data = data.rename(columns={
            'BbAv>2.5': 'over_2_5_goals_odds',
            'BbAv<2.5': 'under_2_5_goals_odds'
        })
        data['home_win_odds'] = np.round((data['B365H'] + data['BWH'] + data['IWH'] + data['PSH'] + data['WHH'] + data['VCH'])/ 6,2)
        data['draw_odds'] = np.round((data['B365D'] + data['BWD'] + data['IWD'] + data['PSD'] + data['WHD'] + data['VCD']) / 6,2)
        data['away_win_odds'] = np.round((data['B365A'] + data['BWA'] + data['IWA'] + data['PSA'] + data['WHA'] + data['VCA']) / 6,2)
    
    data['date_time'] = data['Date'] + ' ' + data['Time'] if 'Time' in data.columns else data['Date'] + ' ' + '00:00'
    data = data[cols]
    data['date_time'] = pd.to_datetime(data['date_time'], format='%d/%m/%Y %H:%M')
    base_data = pd.concat([base_data, data], axis=0)

base_data = base_data.replace(to_replace=teams_replacement_dict)
base_data['idx'] = base_data.apply(lambda row: f"{str(row['date_time'])[:10]}_{row['home_team']}_{row['away_team']}", axis=1)

if not os.path.exists(os.path.join(extracted_data_output_path, 'base_data.csv')):
    base_data.to_csv(os.path.join(extracted_data_output_path, 'base_data.csv'), index=False)


C:\Users\gcich\AppData\Local\Temp\ipykernel_14640\2445090583.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  base_data = pd.concat([base_data, data], axis=0)
C:\Users\gcich\AppData\Local\Temp\ipykernel_14640\2445090583.py:83: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  base_data = base_data.replace(to_replace=teams_replacement_dict)


Scrape xG data

In [7]:
def get_page_html(url):
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3', 'Accept-Language': 'en-US,en;q=0.9'})
    if response.status_code == 200:
        return response.text
    else:
        raise Exception(f'Incorrect status returned: {response.status_code}')
    # else:
    #     raise Exception("This link is not allowed for scraping by robots.txt")
    
def get_elements(page_html, select_phrase):
    """get elements of the website using soup.select method."""
    soup = BeautifulSoup(page_html, 'html.parser')
    return soup.select(select_phrase)

def getPattern(tag_list, pattern, limit, output_datatype = 'int'):
    """searches for given pattern in input_list"""
    output_list = []
    for i in range(limit):
        match = re.search(pattern,tag_list[i].text)
        if match:
            if output_datatype == 'int':
                match = int(match.group())
            elif output_datatype == 'float':
                match = float(match.group())
                
            output_list.append(match)
        else:
            output_list.append(0)
    return output_list
    

In [ ]:
url_list = [
    'https://fbref.com/en/comps/9/2017-2018/schedule/2017-2018-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2018-2019/schedule/2018-2019-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2019-2020/schedule/2019-2020-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2020-2021/schedule/2020-2021-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2021-2022/schedule/2021-2022-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2022-2023/schedule/2022-2023-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2023-2024/schedule/2023-2024-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures'
]
html_list = []
for url in url_list:
    html_list.append(get_page_html(url))
    time.sleep(2)

html_list

['    \n      \n<!DOCTYPE html>\n<html data-version="klecko-" data-root="/home/fb/deploy/www/base" lang="en" class="no-js" >\n<head>\n    <meta charset="utf-8">\n    <meta http-equiv="x-ua-compatible" content="ie=edge">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=2.0" />\n    <link rel="dns-prefetch" href="https://cdn.ssref.net/req/202505152" />\n<script>\n/* https://docs.osano.com/hc/en-us/articles/22469433444372-Google-Consent-Mode-v2  */\n  window.dataLayer = window.dataLayer ||[];\n      function gtag(){dataLayer.push(arguments);}\n      gtag(\'consent\',\'default\',{\n        \'ad_storage\':\'denied\',\n        \'analytics_storage\':\'denied\',\n        \'ad_user_data\':\'denied\',\n        \'ad_personalization\':\'denied\',\n        \'personalization_storage\':\'denied\',\n        \'functionality_storage\':\'granted\',\n        \'security_storage\':\'granted\',\n        \'wait_for_update\': 500\n      });\n      gtag("set", "ads_data_r

In [4]:
def process_div_stats(div, stats_dict):
    stats_string = str(div).replace('<div>', '').replace('</div>', ',')
    stats_list = stats_string.split('\n')[2:-1]
    for stat in stats_list:
        stat_values = stat.split(',')
        stats_dict[f'home_{stat_values[1]}'] = stat_values[0]
        stats_dict[f'away_{stat_values[1]}'] = stat_values[2]
    
    return stats_dict

def process_subpage_stats(subpage):
    tables = get_elements(subpage, 'table')
    stats_dict = dict()

    possesion_tds = tables[2].find_all('tr')[2].find_all('td')
    possesion_values = [td.find('strong').text for td in possesion_tds]

    stats_dict['home_possesion'] = int(possesion_values[0][:-1])
    stats_dict['away_possesion'] = int(possesion_values[1][:-1])

    passes_tds = tables[2].find_all('tr')[4].find_all('td')
    passes_accuracy_values = [td.find('strong').text for td in passes_tds]

    stats_dict['home_passes_acc'] = int(passes_accuracy_values[0][:-1])
    stats_dict['away_passes_acc'] = int(passes_accuracy_values[1][:-1])

    passes_values = [td.find_all('div')[1].find(string=True, recursive=False).split('\xa0—\xa0') for td in passes_tds]
    # stats_dict['home_passes_acc'] = int(passes_accuracy_values[0][:-1])
    stats_dict['home_passes_total'] = passes_values[0][0].split(' of ')[1]
    stats_dict['home_passes_completed'] = passes_values[0][0].split(' of ')[0]
    stats_dict['away_passes_total'] = passes_values[1][1].split(' of ')[1]
    stats_dict['away_passes_completed'] = passes_values[1][1].split(' of ')[0]

    stats_extra = get_elements(subpage, "div#team_stats_extra")[0]

    stats_extra_divs = stats_extra.find_all('div')
    div_1 = stats_extra_divs[0]
    div_2 = stats_extra_divs[16]
    div_3 = stats_extra_divs[32]

    stats_dict = process_div_stats(div_1, stats_dict)
    stats_dict = process_div_stats(div_2, stats_dict)
    stats_dict =process_div_stats(div_3, stats_dict)

    return stats_dict



In [ ]:
detailed_data = pd.DataFrame(
    columns=[
        'season', 'gameweek', 'dayofweek', 'datetime', 
        'team', 'opponent', 'home_game', 'xG', 'xGA',
        'possesion', 'opponent_possesion', 'passes_accuracy', 
        'opponent_passes_accuracy', 'passes_total', 'passes_completed',
        'opponent_passes_total', 'opponent_passes_completed',
        'crosses', 'opponent_crosses', 'touches', 'opponent_touches'
    ])
year = 2017

for html in html_list:
    season = f"{year}/{year+1}"
    
    table_id = f"table#sched_{year}-{year+1}_9_1 tr"
    rows = get_elements(html, table_id)
    
    for row in rows[1:]:
          
        gw = row.find('th').text
        if gw == '':
            continue
        
        cols = row.find_all('td')
        
        content = [col.text for col in cols[:-2]]
        # print(content)
        
        #list have columns: gameweek, dayofweek, datetime, team, opponent, home/away, xG, xGA
        home_row = [season, gw, content[0], f'{content[1]} {content[2]}', content[3], content[7], 1, content[4], content[6]]
        away_row = [season, gw, content[0], f'{content[1]} {content[2]}', content[7], content[3], 0, content[6], content[4]]

        report_tag = cols[-2].find('a')
        report_link = f"https://fbref.com{report_tag.get('href')}"
        
        subpage = get_page_html(report_link)
        sts = process_subpage_stats(subpage)
        home_row.extend([
            sts['home_possesion'],
            sts['away_possesion'],
            sts['home_passes_acc'],
            sts['away_passes_acc'],
            sts['home_passes_total'],
            sts['home_passes_completed'],
            sts['away_passes_total'],
            sts['away_passes_completed'],    
            sts['home_Crosses'],
            sts['away_Crosses'],
            sts['home_Touches'],
            sts['away_Touches']
        ])
        away_row.extend([
            sts['away_possesion'],
            sts['home_possesion'],
            sts['away_passes_acc'],
            sts['home_passes_acc'],
            sts['away_passes_total'],
            sts['away_passes_completed'],  
            sts['home_passes_total'],
            sts['home_passes_completed'],
            sts['away_Crosses'],
            sts['home_Crosses'],
            sts['away_Touches'],
            sts['home_Touches']
        ])
        
        detailed_data.loc[len(detailed_data)] = home_row
        detailed_data.loc[len(detailed_data)] = away_row
        detailed_data['datetime'] = pd.to_datetime(detailed_data['datetime'])
         
        time.sleep(4)
        
    print(f'year {year} completed')
    year += 1
    
detailed_data.index = detailed_data.index + 1
detailed_data = detailed_data.replace(teams_replacement_dict)
detailed_data['idx'] = detailed_data.apply(lambda row: 
    f"{str(row['datetime'])[:10]}_{row['team']}_{row['opponent']}"
    if row['home_game'] == 1 else
    f"{str(row['datetime'])[:10]}_{row['opponent']}_{row['team']}"
    , axis=1)


year 2021 completed
year 2022 completed
year 2023 completed
year 2024 completed


OSError: Cannot save file into a non-existent directory: 'processed_data'

In [ ]:
detailed_data.to_csv(os.path.join(extracted_data_output_path, 'detailed_data.csv'), index=False)
detailed_data

Loading ITables v2.4.0 from the init_notebook_mode cell... (need help?)


Add cups information to the dataset

In [53]:
detailed_data = pd.read_csv(os.path.join(extracted_data_output_path, 'detailed_data.csv'))
detailed_data['comp_before'] = 0
detailed_data['comp_before_name'] = None
detailed_data['comp_before_date'] = None
detailed_data['comp_after'] = 0
detailed_data['comp_after_name'] = None
detailed_data['comp_after_date'] = None

detailed_data

Loading ITables v2.4.0 from the init_notebook_mode cell... (need help?)


In [48]:
league_stats_urls = [
    'https://fbref.com/en/comps/9/2017-2018/2017-2018-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2018-2019/2018-2019-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2019-2020/2019-2020-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2020-2021/2020-2021-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2021-2022/2021-2022-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2022-2023/2022-2023-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2023-2024/2023-2024-Premier-League-Stats',
    'https://fbref.com/en/comps/9/Premier-League-Stats',
]
team_seasons_df = pd.DataFrame(columns=['season', 'team', 'url'])
year = 2017

for league_url in league_stats_urls:
    season = f"{year}/{year+1}"
    season_html = get_page_html(league_url)
    table_id = f"table#results{year}-{year+1}91_overall tr"
    rows = get_elements(season_html, table_id)

    for row in rows[1:]:
        td = row.find('td')
        team_name = td.find('a').text.strip()
        team_season_url = f"https://fbref.com{td.find('a').get('href')}"
        team_seasons_df.loc[len(team_seasons_df)] = [season, team_name, team_season_url]
     
    year += 1    
    time.sleep(4)
    print(f'Season {season} completed')
team_seasons_df = team_seasons_df.replace(to_replace=teams_replacement_dict)
team_seasons_df


Season 2017/2018 completed
Season 2018/2019 completed
Season 2019/2020 completed
Season 2020/2021 completed
Season 2021/2022 completed
Season 2022/2023 completed
Season 2023/2024 completed
Season 2024/2025 completed


Loading ITables v2.4.0 from the init_notebook_mode cell... (need help?)


In [62]:
# for every team, analyze each season season to get cup matches before and after premier league matches 
for idx, row in team_seasons_df.iterrows():

    team = row['team']
    season = row['season']
    url = row['url']
    
    subpage_html = get_page_html(url)
    table_id = f'table#matchlogs_for tr'
    rows = get_elements(subpage_html, table_id)
    gw_no = 0
    previous_comp = None
    previous_comp_date = None
    # rows in that table are matches played by the team in that season
    for row in rows[1:]:
        th = row.find('th')
        comp_date = th.find('a').text.strip()

        tds = row.find_all('td')
        comp_name = tds[1].find('a').text.strip()
        
        round_name = tds[2].find('a').text.strip()

        # if comp_name is any other comp, rows for previous and next gw are updated
        if comp_name != 'Premier League':
            if gw_no > 0 and previous_comp == 'Premier League':
                idx = (detailed_data['team'] == team) & \
                        (detailed_data['season'] == season) & \
                        (detailed_data['gameweek'] == gw_no)
                detailed_data.loc[idx, 'comp_after'] = 1
                detailed_data.loc[idx, 'comp_after_name'] = comp_name
                detailed_data.loc[idx, 'comp_after_date'] = comp_date
                
        # if comp is premier league, check if previous comp was different
        else:
            gw_no = int(round_name.split(' ')[-1])
            if previous_comp != 'Premier League' and previous_comp is not None:
                idx = (detailed_data['team'] == team) & \
                    (detailed_data['season'] == season) & \
                    (detailed_data['gameweek'] == gw_no)
                print(previous_comp, previous_comp_date)   
                detailed_data.loc[idx, 'comp_before'] = 1
                detailed_data.loc[idx, 'comp_before_name'] = previous_comp
                detailed_data.loc[idx, 'comp_before_date'] = previous_comp_date
            
        previous_comp = comp_name
        previous_comp_date = comp_date
    time.sleep(4)


Champions Lg 2017-09-13
EFL Cup 2017-09-20
Champions Lg 2017-09-26
Champions Lg 2017-10-17
EFL Cup 2017-10-24
Champions Lg 2017-11-01
Champions Lg 2017-11-21
Champions Lg 2017-12-06
EFL Cup 2017-12-19
EFL Cup 2018-01-09
FA Cup 2018-01-28
EFL Cup 2018-02-25
Champions Lg 2018-03-07
Champions Lg 2018-04-04
Champions Lg 2018-04-10
Super Cup 2017-08-08
Champions Lg 2017-09-12
EFL Cup 2017-09-20
Champions Lg 2017-09-28
Champions Lg 2017-10-18
EFL Cup 2017-10-24
Champions Lg 2017-10-31
Champions Lg 2017-11-22
Champions Lg 2017-12-05
EFL Cup 2017-12-20
FA Cup 2018-01-05
FA Cup 2018-01-26
Champions Lg 2018-02-21
FA Cup 2018-03-17
FA Cup 2018-04-21
Champions Lg 2017-09-13
EFL Cup 2017-09-19
Champions Lg 2017-09-26
Champions Lg 2017-10-17
EFL Cup 2017-10-25
Champions Lg 2017-11-01
Champions Lg 2017-11-21
Champions Lg 2017-12-06
FA Cup 2018-01-07
FA Cup 2018-01-27
FA Cup 2018-02-07
FA Cup 2018-02-18
FA Cup 2018-02-28
Champions Lg 2018-03-07
FA Cup 2018-03-17
FA Cup 2018-04-21
Champions Lg 2017-08-

In [67]:
detailed_data['comp_before_date'] = pd.to_datetime(detailed_data['comp_before_date'], errors='coerce')
detailed_data['comp_after_date'] = pd.to_datetime(detailed_data['comp_after_date'], errors='coerce')
detailed_data
        

Loading ITables v2.4.0 from the init_notebook_mode cell... (need help?)


In [68]:
detailed_data.to_csv(os.path.join(extracted_data_output_path, 'detailed_data_with_cups.csv'), index=False)

Add manager information to the dataset

In [ ]:
url_list = [
    'https://fbref.com/en/comps/9/2017-2018/schedule/2017-2018-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2018-2019/schedule/2018-2019-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2019-2020/schedule/2019-2020-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2020-2021/schedule/2020-2021-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2021-2022/schedule/2021-2022-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2022-2023/schedule/2022-2023-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/2023-2024/schedule/2023-2024-Premier-League-Scores-and-Fixtures',
    'https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures'
]
html_list = []
for url in url_list:
    html_list.append(get_page_html(url))
    time.sleep(2)

html_list

['    \n      \n<!DOCTYPE html>\n<html data-version="klecko-" data-root="/home/fb/deploy/www/base" lang="en" class="no-js" >\n<head>\n    <meta charset="utf-8">\n    <meta http-equiv="x-ua-compatible" content="ie=edge">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=2.0" />\n    <link rel="dns-prefetch" href="https://cdn.ssref.net/req/202505152" />\n<script>\n/* https://docs.osano.com/hc/en-us/articles/22469433444372-Google-Consent-Mode-v2  */\n  window.dataLayer = window.dataLayer ||[];\n      function gtag(){dataLayer.push(arguments);}\n      gtag(\'consent\',\'default\',{\n        \'ad_storage\':\'denied\',\n        \'analytics_storage\':\'denied\',\n        \'ad_user_data\':\'denied\',\n        \'ad_personalization\':\'denied\',\n        \'personalization_storage\':\'denied\',\n        \'functionality_storage\':\'granted\',\n        \'security_storage\':\'granted\',\n        \'wait_for_update\': 500\n      });\n      gtag("set", "ads_data_r

In [9]:
def process_subpage_manager(subpage):
    div_scorebox = get_elements(subpage, 'div.scorebox')

    stats_dict = dict()

    home_div = div_scorebox[0].find_all('div', recursive=False)[0]

    manager_home_div = home_div.find('div', class_='datapoint')
    manager_home = manager_home_div.find_all(text=True, recursive=False)[0].strip()[2:]
    stats_dict['home_manager'] = manager_home
    
    away_div = div_scorebox[0].find_all('div', recursive=False)[1]

    manager_away_div = away_div.find('div', class_='datapoint')
    manager_away = manager_away_div.find_all(text=True, recursive=False)[0].strip()[2:]
    stats_dict['away_manager'] = manager_away

    return stats_dict

In [10]:

manager_data = pd.DataFrame(
    columns=[
        'season', 'gameweek', 'datetime', 'team', 'opponent', 'home_game', 'manager'
    ])
year = 2021

for html in html_list:
    season = f"{year}/{year+1}"
    
    table_id = f"table#sched_{year}-{year+1}_9_1 tr"
    rows = get_elements(html, table_id)
    i = 0
    for row in rows[1:]:
          
        gw = row.find('th').text
        if gw == '':
            continue
        
        cols = row.find_all('td')
        
        content = [col.text for col in cols[:-2]]
        # print(content)
        
        #list have columns: gameweek, dayofweek, datetime, team, opponent, home/away, xG, xGA
        home_row = [season, gw, f'{content[1]} {content[2]}', content[3], content[7], 1]
        away_row = [season, gw, f'{content[1]} {content[2]}', content[7], content[3], 0]

        report_tag = cols[-2].find('a')
        report_link = f"https://fbref.com{report_tag.get('href')}"
        
        subpage = get_page_html(report_link)
        sts = process_subpage_manager(subpage)
        home_row.extend([
            sts['home_manager'],
        ])
        away_row.extend([
            sts['away_manager'],
        ])
        
        manager_data.loc[len(manager_data)] = home_row
        manager_data.loc[len(manager_data)] = away_row
        
        time.sleep(4)
        
    print(f'year {year} completed')
    year += 1

manager_data['datetime'] = pd.to_datetime(manager_data['datetime']) 
manager_data.index = manager_data.index + 1
manager_data = manager_data.replace(teams_replacement_dict)
manager_data['idx'] = manager_data.apply(lambda row: 
    f"{str(row['datetime'])[:10]}_{row['team']}_{row['opponent']}"
    if row['home_game'] == 1 else
    f"{str(row['datetime'])[:10]}_{row['opponent']}_{row['team']}"
    , axis=1)

manager_data = manager_data[['idx', 'team', 'manager']]

year 2021 completed
year 2022 completed
year 2023 completed
year 2024 completed


In [11]:
manager_data

,idx,team,manager
1,2021-08-13_Brentford_Arsenal,Brentford,Thomas Frank
2,2021-08-13_Brentford_Arsenal,Arsenal,Mikel Arteta
3,2021-08-14_Manchester United_Leeds,Manchester United,Ole Gunnar Solskjær
4,2021-08-14_Manchester United_Leeds,Leeds,Marcelo Bielsa
5,2021-08-14_Everton_Southampton,Everton,Rafael Benítez
...,...,...,...
3036,2025-05-25_Newcastle_Everton,Everton,David Moyes
3037,2025-05-25_Bournemouth_Leicester,Bournemouth,Andoni Iraola
3038,2025-05-25_Bournemouth_Leicester,Leicester,Ruud van Nistelrooy
3039,2025-05-25_Tottenham_Brighton,Tottenham,Ange Postecoglou


In [21]:
manager_data_1 = pd.read_csv(os.path.join(extracted_data_output_path, 'manager_data_to_2022-02-09.csv'))
manager_data_1 = manager_data_1[manager_data_1['season'] != '2021/2022']
manager_data_1 = manager_data_1.replace(teams_replacement_dict)
manager_data_1['idx'] = manager_data_1.apply(lambda row: 
    f"{str(row['datetime'])[:10]}_{row['team']}_{row['opponent']}"
    if row['home_game'] == 1 else
    f"{str(row['datetime'])[:10]}_{row['opponent']}_{row['team']}"
    , axis=1)
menager_data_complete = pd.concat([manager_data_1[['idx', 'team', 'manager']], manager_data], axis=0, ignore_index=True)
menager_data_complete.to_csv(os.path.join(extracted_data_output_path, 'manager_data.csv'), index=False)
menager_data_complete

,idx,team,manager
0,2017-08-11_Arsenal_Leicester,Arsenal,Arsène Wenger
1,2017-08-11_Arsenal_Leicester,Leicester,Craig Shakespeare
2,2017-08-12_Watford_Liverpool,Watford,Marco Silva
3,2017-08-12_Watford_Liverpool,Liverpool,Jürgen Klopp
4,2017-08-12_Everton_Stoke,Everton,Ronald Koeman
...,...,...,...
6075,2025-05-25_Newcastle_Everton,Everton,David Moyes
6076,2025-05-25_Bournemouth_Leicester,Bournemouth,Andoni Iraola
6077,2025-05-25_Bournemouth_Leicester,Leicester,Ruud van Nistelrooy
6078,2025-05-25_Tottenham_Brighton,Tottenham,Ange Postecoglou


In [20]:
menager_data_complete = pd.read_csv(os.path.join(extracted_data_output_path, 'manager_data.csv'))
menager_data_complete

,idx,team,manager
0,2017-08-11_Arsenal_Leicester City,Arsenal,Arsène Wenger
1,2017-08-11_Arsenal_Leicester City,Leicester City,Craig Shakespeare
2,2017-08-12_Watford_Liverpool,Watford,Marco Silva
3,2017-08-12_Watford_Liverpool,Liverpool,Jürgen Klopp
4,2017-08-12_Everton_Stoke City,Everton,Ronald Koeman
...,...,...,...
6075,2025-05-25_Newcastle_Everton,Everton,David Moyes
6076,2025-05-25_Bournemouth_Leicester,Bournemouth,Andoni Iraola
6077,2025-05-25_Bournemouth_Leicester,Leicester,Ruud van Nistelrooy
6078,2025-05-25_Tottenham_Brighton,Tottenham,Ange Postecoglou
